# ⚡ AI Flashcard Generator + Chatbot 🗨


In [9]:
!pip install -q "gradio==3.50.2" groq PyMuPDF
import importlib, gradio
importlib.reload(gradio)
print("Gradio version:", gradio.__version__)

Gradio version: 3.50.2


## Imports

In [10]:
import gradio as gr
from groq import Groq
import fitz
import json
import re
from google.colab import userdata

## Configure API Key

In [11]:
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

client = Groq(api_key=GROQ_API_KEY)

MODELS = {
    "Llama 3.1 8B  (Fast)"   : "llama-3.1-8b-instant",
    "Llama 3.3 70B (Smart)"  : "llama-3.3-70b-versatile",
    "Llama 4 Scout (Latest)" : "meta-llama/llama-4-scout-17b-16e-instruct",
}

print("Groq ready!")

Groq ready!


## Chat Logic

In [12]:
def chat_with_groq(message, history, model_name):
    model_id = MODELS.get(model_name, "llama-3.1-8b-instant")
    messages = [{"role": "system", "content": "You are a helpful AI assistant. Be concise and clear."}]
    for human, assistant in history:
        messages.append({"role": "user",      "content": human})
        messages.append({"role": "assistant", "content": assistant})
    messages.append({"role": "user", "content": message})
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=messages,
            temperature=0.7,
            max_tokens=2048,
        )
        return response.choices[0].message.content
    except Exception as e:
        return "Error: " + str(e)

def respond(message, history, model_name):
    if not message.strip():
        return history, ""
    reply = chat_with_groq(message, history, model_name)
    history = history + [(message, reply)]
    return history, ""

## Flashcard Logic

In [13]:
def extract_pdf_text(pdf_file):
    if pdf_file is None:
        return ""
    doc = fitz.open(pdf_file.name)
    pages = [page.get_text() for page in doc]
    doc.close()
    return "\n\n".join(pages)


def build_prompt(content, num_cards, difficulty):
    prompt  = "You are an expert tutor. Generate exactly " + str(num_cards) + " flashcards from the text below.\n\n"
    prompt += "Difficulty: " + difficulty + "\n"
    prompt += "- Easy   -> definitions, basic recall\n"
    prompt += "- Medium -> application, cause-effect\n"
    prompt += "- Hard   -> synthesis, edge cases\n\n"
    prompt += "Return ONLY a valid JSON array. No markdown, no extra text.\n"
    prompt += "Keys per object: \"question\", \"answer\", \"hint\"\n\n"
    prompt += "TEXT:\n" + content[:8000]
    return prompt


def parse_cards(raw):
    raw = re.sub(r"```(?:json)?", "", raw).strip().rstrip("`").strip()
    start = raw.find("[")
    end   = raw.rfind("]")
    if start == -1 or end == -1:
        raise ValueError("No JSON array found. Raw: " + raw[:300])
    return json.loads(raw[start:end + 1])


def generate_flashcards(text_input, pdf_file, num_cards, difficulty, model_name):
    model_id = MODELS.get(model_name, "llama-3.1-8b-instant")
    content  = text_input.strip()
    if pdf_file is not None:
        content = (content + "\n\n" + extract_pdf_text(pdf_file)).strip()
    if not content:
        return [], "Please paste text, or upload a PDF."
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": build_prompt(content, num_cards, difficulty)}],
            temperature=0.7,
            max_tokens=4096,
        )
        cards = parse_cards(response.choices[0].message.content)
    except json.JSONDecodeError as e:
        return [], "JSON parse error: " + str(e)
    except Exception as e:
        return [], "Error: " + str(e)
    if not cards:
        return [], "Model returned 0 cards. Try different input."
    return cards, str(len(cards)) + " flashcard(s) generated!"


def render_card(cards, index):
    if not cards:
        return "No cards yet.", "", "", ""
    card = cards[index]
    nav  = "Card " + str(index + 1) + " / " + str(len(cards))
    return nav, card["question"], card["answer"], card["hint"]

def prev_card(cards, index):
    if not cards:
        return index, "—", "—", "—", "—"
    index = (index - 1) % len(cards)
    nav, q, a, h = render_card(cards, index)
    return index, nav, q, a, h

def next_card(cards, index):
    if not cards:
        return index, "—", "—", "—", "—"
    index = (index + 1) % len(cards)
    nav, q, a, h = render_card(cards, index)
    return index, nav, q, a, h

def on_generate(text, pdf, num, diff, model_name):
    cards, msg = generate_flashcards(text, pdf, num, diff, model_name)
    if not cards:
        return cards, 0, msg, "—", "—", "—", "—"
    nav, q, a, h = render_card(cards, 0)
    return cards, 0, msg, nav, q, a, h

def export_cards(cards):
    if not cards:
        return None
    lines = []
    for i, c in enumerate(cards, 1):
        lines.append("--- Card " + str(i) + " ---")
        lines.append("Q: " + c["question"])
        lines.append("A: " + c["answer"])
        lines.append("Hint: " + c["hint"])
        lines.append("")
    path = "/tmp/flashcards.txt"
    with open(path, "w") as f:
        f.write("\n".join(lines))
    return path

## UI

In [14]:
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;700;800&family=DM+Mono:ital,wght@0,400;0,500;1,400&display=swap');

body, .gradio-container { background: #0a0a0a !important; }
.gradio-container {
    font-family: 'DM Mono', monospace !important;
    color: #e8e4d9 !important;
    max-width: 980px !important;
    margin: 0 auto !important;
}

/* Header */
#app-header {
    text-align: center;
    padding: 2rem 0 1rem;
    border-bottom: 1px solid #1a1a1a;
    margin-bottom: 1.2rem;
}
#app-header h1 {
    font-family: 'Syne', sans-serif;
    font-size: 2.4rem;
    font-weight: 800;
    background: linear-gradient(135deg, #007bff 0%, #00c8ff 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    margin: 0 0 0.3rem;
    letter-spacing: -0.03em;
}
#app-header p { color: #444; font-size: 0.78rem; margin: 0; }

/* New Mode Selection Buttons */
#mode-selection-row {
    display: flex;
    justify-content: center;
    gap: 1rem;
    margin-bottom: 1.5rem;
}
.mode-button {
    background: #1a1a1a !important;
    border: 1px solid #2a2a2a !important;
    color: #e8e4d9 !important;
    border-radius: 8px !important;
    padding: 0.6rem 1.5rem !important;
    font-family: 'Syne', sans-serif !important;
    font-size: 0.9rem !important;
    font-weight: 700 !important;
    cursor: pointer;
    transition: all 0.2s ease-in-out;
}
.mode-button:hover { background: #2a2a2a !important; border-color: #3a3a3a !important; }
.mode-button-active {
    background: linear-gradient(135deg, #8A2BE2 0%, #00FF7F 100%) !important; /* Purple to Green gradient */
    border-color: #8A2BE2 !important;
    color: white !important;
    box-shadow: 0 4px 10px rgba(138, 43, 226, 0.3); /* Adjust shadow to match purple */
}

/* Gradio buttons */
#btn-toggle {
    background: #1a1a1a !important;
    border: 1px solid #2a2a2a !important;
    color: #e8e4d9 !important;
    border-radius: 20px !important;
    padding: 0.4rem 1.2rem !important;
    font-family: 'DM Mono', monospace !important;
    font-size: 0.8rem !important;
}
#gen-btn {
    background: linear-gradient(135deg, #ff4444, #ff8800) !important;
    color: white !important;
    font-weight: 700 !important;
    border: none !important;
    border-radius: 8px !important;
}
#send-btn {
    background: #ff4444 !important;
    color: white !important;
    font-weight: 700 !important;
    border: none !important;
    border-radius: 8px !important;
}
#clear-btn {
    background: #1a1a1a !important;
    color: #666 !important;
    border: 1px solid #2a2a2a !important;
    border-radius: 8px !important;
}

/* Card display */
#q-box {
    font-family: 'Syne', sans-serif;
    font-size: 1.1rem;
    font-weight: 700;
    color: #ff4444;
    border-left: 3px solid #ff4444;
    padding-left: 1rem;
    margin-bottom: 1.2rem;
    line-height: 1.5;
}
#a-box {
    background: #141414;
    border-radius: 8px;
    padding: 1rem;
    color: #e8e4d9;
    font-size: 0.9rem;
    margin-bottom: 0.8rem;
}
#hint-box {
    font-style: italic;
    color: #555;
    font-size: 0.78rem;
    border-top: 1px solid #1a1a1a;
    padding-top: 0.6rem;
}
#card-nav { color: #555; font-size: 0.75rem; margin-bottom: 0.8rem; }

/* Inputs */
.gr-textbox textarea, .gr-textbox input {
    background: #111 !important;
    border: 1px solid #222 !important;
    color: #e8e4d9 !important;
    border-radius: 8px !important;
    font-family: 'DM Mono', monospace !important;
}
label span { color: #555 !important; font-size: 0.75rem !important; }
.gr-dropdown select { background: #111 !important; color: #e8e4d9 !important; border: 1px solid #222 !important; }
"""

In [15]:
with gr.Blocks(css=CSS, title="Flashcard + Chatbot") as demo:

    cards_state = gr.State([])
    index_state = gr.State(0)
    mode_state  = gr.State("chat")

    gr.HTML("""
    <div id="app-header">
      <h1>AI Flashcard Generator + Chatbot</h1>
      <p>Groq-powered · Switch between Chat and Flashcard mode</p>
    </div>
    """)

    # New mode selection buttons
    with gr.Row(elem_id="mode-selection-row"):
        chat_button = gr.Button("💬 Chatbot", elem_id="chat-mode-btn", elem_classes=["mode-button", "mode-button-active"])
        flash_button = gr.Button("🃏 Flashcards", elem_id="flash-mode-btn", elem_classes=["mode-button"])

    model_selector = gr.Dropdown(
        choices=list(MODELS.keys()),
        value="Llama 3.1 8B  (Fast)",
        label="Model",
        interactive=True,
    )

    # CHAT PANEL
    with gr.Column(visible=True) as chat_panel:
        chatbot = gr.Chatbot(height=380, label="")
        with gr.Row():
            chat_input = gr.Textbox(placeholder="Ask anything...", show_label=False, lines=1)
            send_btn   = gr.Button("Send", elem_id="send-btn", scale=0)
        clear_btn = gr.Button("Clear Chat", elem_id="clear-btn")

    # FLASHCARD PANEL
    with gr.Column(visible=False) as flash_panel:
        with gr.Row():
            with gr.Column():
                text_input = gr.Textbox(
                    label="Paste text",
                    placeholder="Lecture notes, article, chapter...",
                    lines=5,
                )
                pdf_input = gr.File(label="Or upload PDF", file_types=[".pdf"])
                with gr.Row():
                    num_slider = gr.Slider(minimum=3, maximum=20, value=8, step=1, label="Cards")
                    diff_radio = gr.Radio(choices=["Easy", "Medium", "Hard"], value="Medium", label="Difficulty")
                gen_btn      = gr.Button("Generate Flashcards", elem_id="gen-btn")
                flash_status = gr.Markdown("")

            with gr.Column():
                card_nav = gr.Markdown("—", elem_id="card-nav")
                card_q   = gr.Markdown("**Question will appear here**", elem_id="q-box")
                with gr.Accordion("Show Answer", open=False):
                    card_a = gr.Markdown("—", elem_id="a-box")
                with gr.Accordion("Show Hint", open=False):
                    card_h = gr.Markdown("—", elem_id="hint-box")
                with gr.Row():
                    prev_btn = gr.Button("← Prev")
                    next_btn = gr.Button("Next →")
                export_btn = gr.Button("Export as .txt")
                file_out   = gr.File(label="Download")

    # Switch mode logic
    def switch_ui_mode(requested_mode):
        if requested_mode == "chat":
            return (
                "chat",
                gr.update(visible=True),  # chat_panel visible
                gr.update(visible=False), # flash_panel hidden
                gr.update(elem_classes=["mode-button", "mode-button-active"]), # Chat button active
                gr.update(elem_classes=["mode-button"])        # Flash button inactive
            )
        else: # requested_mode == "flash"
            return (
                "flash",
                gr.update(visible=False), # chat_panel hidden
                gr.update(visible=True),  # flash_panel visible
                gr.update(elem_classes=["mode-button"]),        # Chat button inactive
                gr.update(elem_classes=["mode-button", "mode-button-active"])  # Flash button active
            )

    # Initialize UI state on load
    demo.load(
        fn=lambda: switch_ui_mode("chat"),
        inputs=[],
        outputs=[mode_state, chat_panel, flash_panel, chat_button, flash_button],
        queue=False
    )

    chat_button.click(
        fn=lambda: switch_ui_mode("chat"),
        outputs=[mode_state, chat_panel, flash_panel, chat_button, flash_button]
    )
    flash_button.click(
        fn=lambda: switch_ui_mode("flash"),
        outputs=[mode_state, chat_panel, flash_panel, chat_button, flash_button]
    )

    # chat events
    send_btn.click(
        fn=respond,
        inputs=[chat_input, chatbot, model_selector],
        outputs=[chatbot, chat_input],
    )
    chat_input.submit(
        fn=respond,
        inputs=[chat_input, chatbot, model_selector],
        outputs=[chatbot, chat_input],
    )
    clear_btn.click(fn=lambda: ([], ""), outputs=[chatbot, chat_input])

    # flashcard events
    gen_btn.click(
        fn=on_generate,
        inputs=[text_input, pdf_input, num_slider, diff_radio, model_selector],
        outputs=[cards_state, index_state, flash_status, card_nav, card_q, card_a, card_h],
    )
    prev_btn.click(fn=prev_card, inputs=[cards_state, index_state], outputs=[index_state, card_nav, card_q, card_a, card_h])
    next_btn.click(fn=next_card, inputs=[cards_state, index_state], outputs=[index_state, card_nav, card_q, card_a, card_h])
    export_btn.click(fn=export_cards, inputs=[cards_state], outputs=[file_out])